# Bronze — Ingestão com marca d'água

Este notebook constrói a camada **bronze**: lê os arquivos Parquet brutos do volume `growdev.bronze.landing`, adiciona a coluna de marca d'água `_ingested_at` (quando o dado foi ingerido) e grava por **append** nas tabelas `growdev.bronze.<tabela>`. No caminho, recapitulamos Python essencial (variáveis, listas, dicionários, funções, f-strings) e a API de DataFrames do PySpark (read, select, filter, withColumn, groupBy, write).

In [0]:
# Recapitulando: variáveis, f-strings, listas e dicionários

CATALOG = "growdev"
SCHEMA = "bronze"
VOLUME_LANDING = f"/Volumes/{CATALOG}/{SCHEMA}/landing"

# Dicionário mapeando cada tabela bronze -> caminho do dado bruto na landing
tabelas_bronze = {
    "account_balances": f"{VOLUME_LANDING}/account_balances",
    "accounts": f"{VOLUME_LANDING}/accounts",
    "customers": f"{VOLUME_LANDING}/customers",
    "loans": f"{VOLUME_LANDING}/loans",
}

print(f"Volume de origem: {VOLUME_LANDING}")
print("Tabelas a ingerir:", list(tabelas_bronze.keys()))

In [0]:
# Recapitulando: funções + API de DataFrames (read, withColumn, write)
from pyspark.sql import functions as F

def ingerir_bronze(nome_tabela, caminho_landing):
    """Lê o Parquet bruto da landing, aplica a marca d'água e faz append na tabela bronze.

    A marca d'água (_ingested_at) registra o momento exato em que cada lote
    de dados entrou na bronze, permitindo reprocessamentos e auditoria.
    """
    tabela_destino = f"{CATALOG}.{SCHEMA}.{nome_tabela}"  # ex.: growdev.bronze.customers

    df_bronze = (
        spark.read.parquet(caminho_landing)          # 1. lê o dado bruto
        .withColumn("_ingested_at", F.current_timestamp())  # 2. marca d'água da ingestão
    )

    df_bronze.write.mode("append").saveAsTable(tabela_destino)  # 3. append na bronze
    return tabela_destino

In [0]:
# Recapitulando: laço for sobre um dicionário + f-strings
for nome_tabela, caminho in tabelas_bronze.items():
    destino = ingerir_bronze(nome_tabela, caminho)
    print(f"OK: {nome_tabela} -> {destino}")

In [0]:
# Recapitulando: read (spark.table), select, filter e groupBy

# 1) Resumo por tabela: total de linhas e lotes de ingestão (cada execução do append é um lote)
for nome_tabela in tabelas_bronze:
    df = spark.table(f"{CATALOG}.{SCHEMA}.{nome_tabela}")
    total = df.count()
    lotes = df.select("_ingested_at").distinct().count()
    print(f"{nome_tabela}: {total} linhas em {lotes} lote(s) de ingestão")

# 2) Exemplo com select + filter: clientes de SP ingeridos na bronze
df_clientes_sp = (
    spark.table(f"{CATALOG}.{SCHEMA}.customers")
    .select("customer_id", "full_name", "state", "_ingested_at")
    .filter(F.col("state") == "SP")
)
display(df_clientes_sp.limit(5))

# 3) Exemplo com groupBy: clientes por estado na bronze
resumo_estados = (
    spark.table(f"{CATALOG}.{SCHEMA}.customers")
    .groupBy("state")
    .count()
    .orderBy(F.desc("count"))
)
display(resumo_estados.limit(10))